# AI Hub Emotion TTS Fine-tuning - Fresh Colab Run

이 노트북은 새 Colab 런타임에서 위에서부터 순서대로 실행하도록 작성되었습니다. 기존 `/content/fish-speech` 세션 상태를 신뢰하지 않고 매번 깨끗하게 다시 구성합니다.

- 데이터/체크포인트/결과 저장소: Colab Pro 계정의 Google Drive
- 데이터 전처리 스크립트: `Gyul-AI-Repository`의 `MaTuna/tts` 브랜치
- Fish Speech 실행 코드: `fishaudio/openaudio-s1-mini` Hugging Face Space의 고정 revision
- 기본 실행 모드: `smoke`

## 0. Runtime

Colab 메뉴에서 `Runtime > Change runtime type > GPU`를 선택하세요. 처음에는 `RUN_MODE = "smoke"` 그대로 끝까지 실행하고, 성공하면 `RUN_MODE = "full"`로 바꾼 뒤 전처리 셀부터 다시 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# First run: smoke. After it succeeds, change to "full" and rerun from dataset preparation.
RUN_MODE = "smoke"  # "smoke" or "full"
assert RUN_MODE in {"smoke", "full"}

DRIVE_ROOT = Path('/content/drive/MyDrive/gyul-ai/emotion-tts')
ARCHIVE_DIR = DRIVE_ROOT / 'archive'
RAW_DIR = DRIVE_ROOT / 'raw'
PROCESSED_ROOT = DRIVE_ROOT / 'processed'
PROTO_ROOT = DRIVE_ROOT / 'protos'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
RESULTS_ROOT = DRIVE_ROOT / 'results'
SAMPLES_ROOT = DRIVE_ROOT / 'generated_samples'

REPO_DIR = Path('/content/Gyul-AI-Repository')
FISH_DIR = Path('/content/fish-speech')
BASE_CKPT = CHECKPOINT_ROOT / 'openaudio-s1-mini'

RUN_PROCESSED_DIR = PROCESSED_ROOT / RUN_MODE
RUN_PROTO_DIR = PROTO_ROOT / RUN_MODE
PROJECT_NAME = f'aihub_emotion_lora_{RUN_MODE}'
RUN_RESULTS_DIR = RESULTS_ROOT / PROJECT_NAME
MERGED_CKPT = CHECKPOINT_ROOT / f'openaudio-s1-mini-aihub-emotion-{RUN_MODE}'

if RUN_MODE == 'smoke':
    SAMPLES_PER_EMOTION = 10
    TRAIN_MAX_STEPS = 20
    CHECKPOINT_EVERY = 20
else:
    SAMPLES_PER_EMOTION = 300
    TRAIN_MAX_STEPS = 1500
    CHECKPOINT_EVERY = 100

for path in [ARCHIVE_DIR, RAW_DIR, PROCESSED_ROOT, PROTO_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, SAMPLES_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('RUN_MODE:', RUN_MODE)
print('samples_per_emotion:', SAMPLES_PER_EMOTION)
print('train_max_steps:', TRAIN_MAX_STEPS)
print('drive_root:', DRIVE_ROOT)
print('base_ckpt:', BASE_CKPT)

## 1. AI Hub 데이터 준비

압축 파일을 쓰는 경우 아래 경로에 업로드하세요.

```text
/content/drive/MyDrive/gyul-ai/emotion-tts/archive/emotion_voice_dataset.zip
```

이미 압축을 풀어 두었다면 이 셀은 압축 해제를 건너뛰고 `raw` 폴더 아래에서 AI Hub 감정별 폴더를 자동 탐색합니다.

In [ ]:
import subprocess

ARCHIVE_PATH = ARCHIVE_DIR / 'emotion_voice_dataset.zip'

if ARCHIVE_PATH.exists():
    print('Extracting archive:', ARCHIVE_PATH)
    subprocess.run(['unzip', '-oq', str(ARCHIVE_PATH), '-d', str(RAW_DIR)], check=True)
else:
    print('Archive not found; assuming raw dataset is already extracted under:', RAW_DIR)

expected_prefixes = ('ang_', 'dis_', 'fea_', 'hap_', 'neu_', 'sad_', 'sur_')
candidates = []
for directory in [RAW_DIR, *RAW_DIR.rglob('*')]:
    if not directory.is_dir():
        continue
    child_names = [child.name for child in directory.iterdir() if child.is_dir()]
    matched = sum(any(name.startswith(prefix) for name in child_names) for prefix in expected_prefixes)
    if matched >= 4:
        candidates.append((matched, directory))

assert candidates, f'AI Hub emotion folders were not found under {RAW_DIR}'
AIHUB_INPUT_DIR = sorted(candidates, key=lambda item: (-item[0], len(str(item[1]))))[0][1]

print('AIHUB_INPUT_DIR:', AIHUB_INPUT_DIR)
for child in sorted(AIHUB_INPUT_DIR.iterdir()):
    if child.is_dir():
        print('-', child.name)

## 2. 프로젝트/모델 코드 설치

`Gyul-AI-Repository`는 데이터 전처리 스크립트만 사용합니다. Fish Speech 실행 코드는 GitHub `main` 대신 OpenAudio S1-mini Space의 고정 revision을 사용합니다.

In [ ]:
REPO_URL = 'https://github.com/novvvv/Gyul-AI-Repository.git'
BRANCH = 'MaTuna/tts'

%cd /content
!rm -rf "{REPO_DIR}"
!git clone --branch "{BRANCH}" "{REPO_URL}" "{REPO_DIR}"
%cd /content/Gyul-AI-Repository
!git status --short --branch
!test -f scripts/prepare_aihub_emotion_dataset.py
!test -f configs/emotion_tags.yaml

In [ ]:
FISH_SPACE_REPO = 'https://huggingface.co/spaces/fishaudio/openaudio-s1-mini'
FISH_SPACE_REV = 'a26769da224cd5055d63c6d5cf6bb6e2075683cd'

%cd /content
!apt-get update -y
!apt-get install -y portaudio19-dev libasound2-dev ffmpeg libsox-dev libsndfile1
!rm -rf "{FISH_DIR}"
!git clone "{FISH_SPACE_REPO}" "{FISH_DIR}"
%cd /content/fish-speech
!git checkout "{FISH_SPACE_REV}"

!python -m pip install --upgrade pip
!python -m pip install -U uv 'huggingface_hub>=0.34.0,<1.0'
!uv pip install --system -e '.[cu126]'
!python -m pip uninstall -y torchvision
!git log -1 --oneline

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print('gpu:', gpu_name)
    print('vram_gb:', round(total_gb, 2))
else:
    raise RuntimeError('GPU runtime is required for this notebook.')

## 3. Base checkpoint 다운로드

`fishaudio/openaudio-s1-mini`는 gated 모델입니다. Hugging Face에서 접근 권한을 승인한 read token으로 로그인하세요.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()
!hf auth whoami

In [ ]:
!hf download fishaudio/openaudio-s1-mini --local-dir "{BASE_CKPT}" --max-workers 1
!find "{BASE_CKPT}" -maxdepth 1 -type f -printf '%f\n' | sort

In [ ]:
%cd /content/fish-speech
from fish_speech.tokenizer import FishTokenizer

required_files = ['codec.pth', 'config.json', 'model.pth', 'special_tokens.json', 'tokenizer.tiktoken']
missing = [name for name in required_files if not (BASE_CKPT / name).exists()]
assert not missing, f'Missing checkpoint files: {missing}'

tokenizer = FishTokenizer.from_pretrained(BASE_CKPT)
semantic_count = len(tokenizer.semantic_id_to_token_id)
print('semantic_count:', semantic_count)
print('semantic_range:', tokenizer.semantic_begin_id, tokenizer.semantic_end_id)
assert semantic_count == 4096, f'Expected 4096 semantic tokens, got {semantic_count}'

## 4. 데이터 전처리

AI Hub 원본 감정 폴더를 Fish Speech fine-tuning 형식인 `*.wav + *.lab`로 변환합니다. `smoke` 모드는 감정당 10개, `full` 모드는 감정당 300개를 사용합니다.

In [ ]:
%cd /content/Gyul-AI-Repository
!python scripts/prepare_aihub_emotion_dataset.py \
  --input-dir "{AIHUB_INPUT_DIR}" \
  --output-dir "{RUN_PROCESSED_DIR}" \
  --samples-per-emotion {SAMPLES_PER_EMOTION} \
  --overwrite

from pathlib import Path
for emotion_dir in sorted(RUN_PROCESSED_DIR.iterdir()):
    if emotion_dir.is_dir():
        wav_count = len(list(emotion_dir.glob('*.wav')))
        lab_count = len(list(emotion_dir.glob('*.lab')))
        print(emotion_dir.name, 'wav:', wav_count, 'lab:', lab_count)

## 5. VQ 추출 및 protobuf 생성

In [ ]:
%cd /content/fish-speech

# T4: 4 권장. L4/A100이면 16으로 올릴 수 있습니다.
VQ_BATCH_SIZE = 4

!python tools/vqgan/extract_vq.py "{RUN_PROCESSED_DIR}" \
  --num-workers 1 \
  --batch-size {VQ_BATCH_SIZE} \
  --config-name modded_dac_vq \
  --checkpoint-path "{BASE_CKPT / 'codec.pth'}"

In [ ]:
%cd /content/fish-speech
!rm -rf "{RUN_PROTO_DIR}"
!python tools/llama/build_dataset.py \
  --input "{RUN_PROCESSED_DIR}" \
  --output "{RUN_PROTO_DIR}" \
  --text-extension .lab \
  --num-workers 4

!rm -rf data/protos
!mkdir -p data
!ln -s "{RUN_PROTO_DIR}" data/protos
!find "{RUN_PROTO_DIR}" -maxdepth 1 -type f -name '*.protos' -print
!ls -la data/protos

## 6. LoRA 학습

`smoke` 모드는 20 step으로 학습 파이프라인 통과 여부만 확인합니다. 성공 후 `full` 모드로 바꾸면 1500 step으로 실행됩니다.

In [ ]:
%cd /content/fish-speech

BATCH_SIZE = 2
GRAD_ACCUM = 8
gpu_name = torch.cuda.get_device_name(0).lower()
PRECISION = '16-mixed' if 't4' in gpu_name else 'bf16-true'

print('project:', PROJECT_NAME)
print('precision:', PRECISION)
print('max_steps:', TRAIN_MAX_STEPS)

!python fish_speech/train.py --config-name text2semantic_finetune \
  project={PROJECT_NAME} \
  pretrained_ckpt_path="{BASE_CKPT}" \
  data.batch_size={BATCH_SIZE} \
  trainer.accumulate_grad_batches={GRAD_ACCUM} \
  trainer.max_steps={TRAIN_MAX_STEPS} \
  trainer.val_check_interval={CHECKPOINT_EVERY} \
  callbacks.model_checkpoint.every_n_train_steps={CHECKPOINT_EVERY} \
  trainer.precision={PRECISION} \
  hydra.run.dir="{RUN_RESULTS_DIR}" \
  +lora@model.model.lora_config=r_8_alpha_16

## 7. LoRA merge

학습 결과 checkpoint를 자동으로 찾아 base checkpoint와 merge합니다.

In [ ]:
%cd /content/fish-speech

ckpt_dir = RUN_RESULTS_DIR / 'checkpoints'
ckpts = sorted(ckpt_dir.glob('*.ckpt'), key=lambda path: path.stat().st_mtime)
print('checkpoint_dir:', ckpt_dir)
for ckpt in ckpts:
    print('-', ckpt)
assert ckpts, f'No .ckpt files found in {ckpt_dir}'

LORA_CKPT = ckpts[-1]
print('selected_lora_ckpt:', LORA_CKPT)
print('merged_output:', MERGED_CKPT)

!rm -rf "{MERGED_CKPT}"
!python tools/llama/merge_lora.py \
  --lora-config r_8_alpha_16 \
  --base-weight "{BASE_CKPT}" \
  --lora-weight "{LORA_CKPT}" \
  --output "{MERGED_CKPT}"

!find "{MERGED_CKPT}" -maxdepth 1 -type f -printf '%f\n' | sort

## 완료

merged checkpoint는 아래 경로에 저장됩니다.

```text
/content/drive/MyDrive/gyul-ai/emotion-tts/checkpoints/openaudio-s1-mini-aihub-emotion-smoke
/content/drive/MyDrive/gyul-ai/emotion-tts/checkpoints/openaudio-s1-mini-aihub-emotion-full
```